<a href="https://colab.research.google.com/github/AlleyAbility/banana-disease-detection/blob/main/Banana_Disease_Phase1_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Banana Disease Classification — Phase 1: Dataset Audit and Leakage-Safe Splits

**Source:** AI4D Lab banana classification notebook. **Classes:** Fusarium, Sigatoka, Healthy.  
**Goal:** verify the original Harvard Dataverse archives, detect corrupted/duplicate/near-duplicate images, and produce deterministic train/validation/test manifests. No model is trained here.

**Research integrity:** A random image split cannot establish independence if the same plant, acquisition session, or source image appears multiple times. Exact-hash grouping prevents byte-identical duplicates crossing splits; perceptual-hash clustering reduces near-duplicate leakage, but cannot guarantee plant-level independence. Inspect flagged groups manually. Never claim external test performance from this internal split.

## 1. Setup and fixed configuration
Run cells in order in Google Colab. Downloading datasets may take time and requires network access.

In [ ]:
%pip -q install pillow pandas scikit-learn matplotlib imagehash tqdm requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 5.8 MB/s eta 0:00:00


In [ ]:
from pathlib import Path
import os, random, json, hashlib, zipfile, shutil, re, time
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError
import imagehash
from tqdm.auto import tqdm
import requests
from sklearn.model_selection import train_test_split

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
ROOT = Path('/content/banana_research')
ARCHIVES = ROOT/'archives'
RAW = ROOT/'raw'
REPORTS = ROOT/'reports'
for d in (ARCHIVES, RAW, REPORTS): d.mkdir(parents=True, exist_ok=True)
SOURCES = {'Sigatoka': 6078711, 'Fusarium': 7071416, 'Healthy': 7077403}
print('Workspace:', ROOT)


Workspace: /content/banana_research


## 2. Download and safely inspect the archives
Files are cached so re-running does not download them again. Archive contents are inspected before extraction. The original notebook extracted only one of the three ZIP files; this version processes all three.

In [ ]:
def download_file(file_id, destination):
    if destination.exists() and destination.stat().st_size > 0:
        print('Cached:', destination.name)
        return
    url = f'https://dataverse.harvard.edu/api/access/datafile/{file_id}'
    temp = destination.with_suffix('.part')
    try:
        with requests.get(url, stream=True, timeout=(30, 180), allow_redirects=True) as response:
            response.raise_for_status()
            with open(temp, 'wb') as f:
                for chunk in response.iter_content(chunk_size=1024*1024):
                    if chunk: f.write(chunk)
        if not zipfile.is_zipfile(temp):
            raise ValueError(f'Download for {file_id} is not a ZIP archive; inspect source URL')
        temp.replace(destination)
    finally:
        if temp.exists(): temp.unlink()

for label, file_id in SOURCES.items():
    download_file(file_id, ARCHIVES/f'{label}.zip')

for label in SOURCES:
    archive = ARCHIVES/f'{label}.zip'
    with zipfile.ZipFile(archive) as z:
        bad = z.testzip()
        if bad: raise ValueError(f'Corrupt ZIP member in {archive}: {bad}')
        entries = [x for x in z.infolist() if not x.is_dir()]
        print(label, 'archive members:', len(entries), 'size MB:', round(archive.stat().st_size/1e6, 1))
        print('Example members:', [x.filename for x in entries[:4]])


Sigatoka archive members: 6144 size MB: 1812.2
Example members: ['BLACK SIGATOKA_2/Image_5764.jpg', '__MACOSX/BLACK SIGATOKA_2/._Image_5764.jpg', 'BLACK SIGATOKA_2/Image_3315.jpg', '__MACOSX/BLACK SIGATOKA_2/._Image_3315.jpg']
Fusarium archive members: 2501 size MB: 1193.7
Example members: ['FUSARIUM WILT_1/.DS_Store', 'FUSARIUM WILT_1/Image_1.jpg', 'FUSARIUM WILT_1/Image_10.jpg', 'FUSARIUM WILT_1/Image_100.jpg']
Healthy archive members: 2769 size MB: 1477.5
Example members: ['HEALTHY_2/.DS_Store', 'HEALTHY_2/Image_3001.jpg', 'HEALTHY_2/Image_3002.jpg', 'HEALTHY_2/Image_3003.jpg']


## 3. Extract images into canonical class folders
The archive directory structure may vary. We use the archive identity as the class label, not folder names inside the archive. Original file names are retained with a deterministic prefix to prevent collisions. Symlinks, traversal paths and non-image members are ignored.

In [ ]:
IMAGE_EXT = {'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
manifest_rows=[]
for label in SOURCES:
    archive=ARCHIVES/f'{label}.zip'
    target=RAW/label
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive) as z:
        members=sorted([m for m in z.infolist() if not m.is_dir()], key=lambda m:m.filename)
        for member in tqdm(members, desc=f'Extract {label}'):
            name=member.filename.replace('\\','/')
            parts=Path(name).parts
            if name.startswith('/') or '..' in parts or Path(name).suffix.lower() not in IMAGE_EXT:
                continue
            if (member.external_attr >> 16) & 0o170000 == 0o120000:
                continue  # ZIP symlink
            safe_name=hashlib.sha256(name.encode()).hexdigest()[:14]+'_'+Path(name).name
            dest=target/safe_name
            if not dest.exists():
                with z.open(member) as source, open(dest,'wb') as out:
                    shutil.copyfileobj(source,out)
            manifest_rows.append({'label':label,'original_member':name,'path':str(dest),'source_id':SOURCES[label]})
raw_manifest=pd.DataFrame(manifest_rows)
print(raw_manifest.groupby('label').size())
print('Total image files:',len(raw_manifest))


Extract Sigatoka:   0%|          | 0/6144 [00:00<?, ?it/s]

Extract Fusarium:   0%|          | 0/2501 [00:00<?, ?it/s]

Extract Healthy:   0%|          | 0/2769 [00:00<?, ?it/s]

label
Fusarium    2500
Healthy     2768
Sigatoka    6142
dtype: int64
Total image files: 11410


## 4. Validate images, compute SHA-256 and perceptual hashes
SHA-256 detects exact byte duplicates. Perceptual hashes (pHash) help identify visually similar images even after resizing/compression. EXIF orientation is normalized before pHash. Corrupt images are excluded and recorded.

In [ ]:
audit=[]
for row in tqdm(raw_manifest.itertuples(index=False), total=len(raw_manifest), desc='Image audit'):
    entry=row._asdict()
    try:
        p=Path(entry['path'])
        with open(p,'rb') as f:
            sha=hashlib.file_digest(f,'sha256').hexdigest()
        with Image.open(p) as im:
            im.verify()
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im)
            w,h=im.size
            rgb=im.convert('RGB')
            ph=str(imagehash.phash(rgb, hash_size=8))
        entry.update({'valid':True,'sha256':sha,'phash':ph,'width':w,'height':h,'error':''})
    except Exception as e:
        entry.update({'valid':False,'sha256':'','phash':'','width':None,'height':None,'error':str(e)[:300]})
    audit.append(entry)
audit_df=pd.DataFrame(audit)
audit_df.to_csv(REPORTS/'image_audit.csv',index=False)
valid=audit_df[audit_df.valid].copy().reset_index(drop=True)
print('Valid:',len(valid),'Corrupt:',(~audit_df.valid).sum())
print(valid.groupby('label').agg(images=('path','size'),min_width=('width','min'),min_height=('height','min')))
print('Exact duplicate image rows:',valid.duplicated('sha256',keep=False).sum())
print('SHA hashes appearing in multiple labels:', (valid.groupby('sha256').label.nunique()>1).sum())


Image audit:   0%|          | 0/11410 [00:00<?, ?it/s]

Valid: 8339 Corrupt: 3071
          images  min_width  min_height
label                                  
Fusarium    2500      288.0       288.0
Healthy     2768     1024.0       768.0
Sigatoka    3071     1024.0       768.0
Exact duplicate image rows: 0
SHA hashes appearing in multiple labels: 0


## 5. Identify near-duplicate groups
For a manageable dataset, use a BK-tree for Hamming distance on 64-bit pHash values. We link images with distance ≤ 4 and build connected components. This is a conservative heuristic, **not proof** of identical source photographs. Review groups before training; raise/lower the threshold after manual inspection.

In [ ]:
class BKNode:
    def __init__(self,value,index): self.value=value; self.indices=[index]; self.children={}
class BKTree:
    def __init__(self):self.root=None
    def insert(self,value,index):
        if self.root is None:self.root=BKNode(value,index);return
        node=self.root
        while True:
            d=(node.value^value).bit_count()
            if d==0:node.indices.append(index);return
            if d not in node.children:node.children[d]=BKNode(value,index);return
            node=node.children[d]
    def search(self,value,radius):
        if self.root is None:return []
        results=[];stack=[self.root]
        while stack:
            node=stack.pop();d=(node.value^value).bit_count()
            if d<=radius:results.extend(node.indices)
            for edge,child in node.children.items():
                if d-radius<=edge<=d+radius:stack.append(child)
        return results

parent=list(range(len(valid)))
def find(x):
    while parent[x]!=x:
        parent[x]=parent[parent[x]];x=parent[x]
    return x
def union(a,b):
    ra,rb=find(a),find(b)
    if ra!=rb:parent[rb]=ra

# Always group byte-identical images, including across class labels.
sha_first={}
for i,sha in enumerate(valid.sha256):
    if sha in sha_first:union(i,sha_first[sha])
    else:sha_first[sha]=i

MAX_PHASH_DISTANCE=4
bkt=BKTree()
for i,ph in enumerate(tqdm(valid.phash,desc='Near-duplicate scan')):
    value=int(ph,16)
    for j in bkt.search(value,MAX_PHASH_DISTANCE):union(i,j)
    bkt.insert(value,i)
valid['group_id']=[find(i) for i in range(len(valid))]
groups=valid.groupby('group_id').agg(n=('path','size'),labels=('label',lambda s:','.join(sorted(set(s)))))
print('Groups:',len(groups),'Multi-image groups:',int((groups.n>1).sum()))
print('Groups spanning multiple labels:',int((groups.labels.str.contains(',')).sum()))
print(groups.sort_values('n',ascending=False).head(10))
valid.to_csv(REPORTS/'audited_manifest.csv',index=False)


## 6. Inspect potentially problematic groups
Cross-class duplicates are especially important: they may indicate conflicting labels. Inspect examples and resolve questionable labels **before** accepting a dataset for publication.

In [ ]:
import matplotlib.pyplot as plt
suspect=groups[(groups.n>1)].sort_values('n',ascending=False).head(6).index
fig,axes=plt.subplots(len(suspect) if len(suspect) else 1,3,figsize=(10,max(3,len(suspect)*2.8)))
axes=np.asarray(axes).reshape(-1,3)
for row,group_id in enumerate(suspect):
    subset=valid[valid.group_id==group_id].head(3)
    for col,(_,item) in enumerate(subset.iterrows()):
        with Image.open(item.path) as im:axes[row,col].imshow(ImageOps.exif_transpose(im).convert('RGB'))
        axes[row,col].set_title(f'{item.label} | group {group_id}',fontsize=9)
for ax in axes.flat:ax.axis('off')
plt.tight_layout();plt.show()
print('If cross-class duplicates exist, resolve their labels and rerun before using these splits.')


## 7. Create deterministic, group-disjoint 70/15/15 splits
This uses a deterministic **greedy group assignment** to approximate per-class 70/15/15 proportions without separating near-duplicate groups. It is not guaranteed to match exact percentages. Large or mixed-label groups can cause imbalance; the diagnostics below must be reviewed. Groups are assigned in descending size, with a fixed-seed tie-breaker.

In [ ]:
RATIOS={'train':0.70,'val':0.15,'test':0.15}
labels=sorted(valid.label.unique())
targets={s:{c:float((valid.label==c).sum())*ratio for c in labels} for s,ratio in RATIOS.items()}
counts={s:Counter() for s in RATIOS}
group_counts=valid.groupby(['group_id','label']).size().unstack(fill_value=0).reindex(columns=labels,fill_value=0)
rng=np.random.default_rng(SEED)
order=list(group_counts.index)
rng.shuffle(order)
order.sort(key=lambda g:-int(group_counts.loc[g].sum()))
assign={}
for group_id in order:
    vector=group_counts.loc[group_id]
    def cost(split):
        # Compare squared normalized deviations over all classes and splits.
        total=0.0
        for s in RATIOS:
            for c in labels:
                candidate=counts[s][c]+(int(vector[c]) if s==split else 0)
                total+=((candidate-targets[s][c])/max(1,targets[s][c]))**2
        return total
    chosen=min(RATIOS,key=cost)
    assign[group_id]=chosen
    counts[chosen].update({c:int(vector[c]) for c in labels})
valid['split']=valid.group_id.map(assign)
print(pd.crosstab(valid.label,valid.split,margins=True))
assert valid.groupby('group_id').split.nunique().max()==1
assert valid.groupby('sha256').split.nunique().max()==1
assert valid.groupby('phash').split.nunique().max()==1
assert set(valid.split)==set(RATIOS),'One or more splits empty: inspect group distribution'
valid.sort_values(['split','label','path']).to_csv(REPORTS/'split_manifest.csv',index=False)
print('Saved:',REPORTS/'split_manifest.csv')


## 8. Reproducible TensorFlow input pipeline
We use the split manifest rather than `validation_split` to prevent accidental reshuffling. The images are decoded and resized; pixel normalization belongs in the model-specific preprocessing stage, which we will implement in Phase 2.

In [ ]:
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
IMG_SIZE=(224,224)
BATCH_SIZE=32
class_names=sorted(valid.label.unique())
class_to_id={name:i for i,name in enumerate(class_names)}

def decode_image(path,label):
    data=tf.io.read_file(path)
    image=tf.io.decode_image(data,channels=3,expand_animations=False)
    image.set_shape([None,None,3])
    image=tf.image.resize(image,IMG_SIZE)
    return image,label

def make_dataset(split,training=False):
    frame=valid[valid.split==split].sort_values('path')
    paths=frame.path.astype(str).tolist()
    labels_numeric=frame.label.map(class_to_id).astype('int32').tolist()
    ds=tf.data.Dataset.from_tensor_slices((paths,labels_numeric))
    if training:ds=ds.shuffle(len(paths),seed=SEED,reshuffle_each_iteration=True)
    return ds.map(decode_image,num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds=make_dataset('train',training=True)
val_ds=make_dataset('val')
test_ds=make_dataset('test')
for name,ds in [('train',train_ds),('val',val_ds),('test',test_ds)]:
    batch=next(iter(ds))
    print(name,'batch shapes:',batch[0].shape,batch[1].shape)
print('Class mapping:',class_to_id)


## 9. Export audit report and splits
No image data is included in the report ZIP. Check dataset license/terms before redistribution; do not upload the raw dataset to GitHub.

In [ ]:
summary={
 'source':'Harvard Dataverse IDs '+str(SOURCES),
 'seed':SEED,'n_archived_images':len(raw_manifest),'n_valid_images':len(valid),
 'n_corrupt':int((~audit_df.valid).sum()),
 'n_unique_sha256':int(valid.sha256.nunique()),
 'n_perceptual_groups':int(valid.group_id.nunique()),
 'cross_label_groups':int((groups.labels.str.contains(',')).sum()),
 'class_counts':{str(k):int(v) for k,v in valid.label.value_counts().items()},
 'split_counts':{str(k):int(v) for k,v in valid.split.value_counts().items()},
 'split_by_class':pd.crosstab(valid.label,valid.split).to_dict(),
 'limitations':['Perceptual hash is heuristic and may join unrelated leaves or miss near-duplicates.',
 'Plant/farm/acquisition-session independence cannot be confirmed without metadata.',
 'An internal holdout is not an external field-validation dataset.']
}
(REPORTS/'quality_summary.json').write_text(json.dumps(summary,indent=2))
print(json.dumps(summary,indent=2))
shutil.make_archive('/content/banana_phase1_reports','zip',REPORTS)
from google.colab import files
files.download('/content/banana_phase1_reports.zip')


## Completion checklist
- [ ] All three archives downloaded and ZIP-tested.
- [ ] Class counts and corruption report reviewed.
- [ ] Cross-class duplicate groups inspected and resolved.
- [ ] Train/val/test distributions reviewed for severe imbalance.
- [ ] No group crosses train/val/test.
- [ ] `banana_phase1_reports.zip` saved outside Colab.

**Next:** Phase 2 model training with model-specific preprocessing, baselines, controlled transfer-learning comparisons, and evaluation on the untouched test split only after model selection.